# YLM: resumable options A/B/C and parity ablation

Select a GPU runtime. Full mode trains A/B/C, task-ID and legacy A on 20,000 QM9 molecules. Smoke mode uses 500 molecules and tiny models.

Data and runs persist in `MyDrive/YLM_runs/options_bc_test`. Rerun setup after a disconnect to reuse cached data and resume the latest completed epoch. Keep training settings unchanged when resuming. Full and smoke experiments have separate names and caches.

Every training run saves profiling under its run directory, including a 20-step trace, synchronized phase timings, throughput, GPU memory, and utilization samples. Timings include profiling overhead. Prior profiling sessions are retained on resume.


**Relaunch (Oct 30):** set the form fields in the first code cell, then Runtime > Run all. Finished runs (with `results.json`) are skipped; an interrupted run resumes from its `latest.pt` (same settings). Order: A (resumes `full_A`), then B, then C. Task-ID and legacy-A runs are optional (`RUN_EXTRAS`). Each run's profiling is under `runs/<name>/profiling/` (summary.json, trace.json, key_averages.txt, gpu_utilization.jsonl).


In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, sys, json, time, shutil

# ---- Form fields (edit here only) ----
FULL = True  # @param {type:"boolean"}
OPTIONS_TO_RUN = 'A,B,C'  # @param {type:"string"}
RUN_EXTRAS = False  # @param {type:"boolean"}
RUN_CPU_BENCH = False  # @param {type:"boolean"}
# --------------------------------------
drive.mount('/content/drive')
REPO = 'YLM'
BRANCH = 'feat/options-bc-oct30'  # @param {type:"string"}
SAVE = Path('/content/drive/MyDrive') / f'{REPO}_runs' / 'options_bc_test'
SAVE.mkdir(parents=True, exist_ok=True)
ROOT = Path('/content/YLM')
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
if not ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/yaniguan/YLM.git', str(ROOT)], check=True)
subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=ROOT, check=True)
subprocess.run(['git', 'checkout', '-B', BRANCH, 'origin/' + BRANCH], cwd=ROOT, check=True)
print(subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=ROOT, capture_output=True, text=True).stdout)
os.chdir(ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'e3nn==0.6.0', 'numpy<2.3',
                'scipy', 'rdkit', 'tblite', 'transformers>=4.40,<5', 'accelerate', 'pytest'], check=True)
import torch
torch.set_num_threads(2)
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
print(torch.cuda.get_device_name(), 'Saving to', SAVE)
# Write results directly to Drive so interrupted runs retain completed checkpoints.
(SAVE / 'runs').mkdir(exist_ok=True)

def run_script(script, *args):
    command = [sys.executable, str(ROOT / 'scripts' / script), *map(str, args)]
    print(' '.join(command), flush=True)
    with (SAVE / 'commands.log').open('a') as log:
        log.write(' '.join(command) + '\n')
    subprocess.run(command, check=True)


## Cached data: scripts 01–03
Build only missing labels, questions, and frozen MiniLM embeddings; reuse Drive copies on reconnect.


In [ ]:
import urllib.request, zipfile
N_MOLECULES = 20000 if FULL else 500
# Full data lives at the fixed run root; smoke data has its own cache.
CACHE = SAVE if FULL else SAVE / 'smoke_data'
CACHE.mkdir(exist_ok=True)
Path('data').mkdir(exist_ok=True)
for name in ['qm9_xtb.pkl', 'tensorqa_v0.json', 'text_minilm.pt']:
    if (CACHE / name).exists():
        shutil.copy2(CACHE / name, Path('data') / name)
if not (CACHE / 'qm9_xtb.pkl').exists():
    archive = Path('data/qm9.zip')
    if not Path('data/gdb9.sdf').exists():
        urllib.request.urlretrieve('https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/molnet_publish/qm9.zip', archive)
        with zipfile.ZipFile(archive) as z:
            z.extractall('data')
    run_script('01_make_labels.py', '--n', N_MOLECULES, '--workers', os.cpu_count() or 2, '--seed', 0)
    shutil.copy2('data/qm9_xtb.pkl', CACHE / 'qm9_xtb.pkl')
if not (CACHE / 'tensorqa_v0.json').exists():
    run_script('02_make_questions.py', '--seed', 0)
    shutil.copy2('data/tensorqa_v0.json', CACHE / 'tensorqa_v0.json')
if not (CACHE / 'text_minilm.pt').exists():
    run_script('03_embed_text.py', '--name', 'minilm')
    shutil.copy2('data/text_minilm.pt', CACHE / 'text_minilm.pt')
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)


## Train and evaluate
Script 04 saves the best validation model and evaluates held-out molecules/wordings. `_o3` results aggregate identity, rotations and their reflections; `_reflection` reports only improper transforms; compare these with rotation-only results. Old/fixed A use the same seed. Increase `MAX_STEPS` and `EPOCHS` for meaningful learning curves; these short runs cannot establish comparative model quality.

In [ ]:
P = 'full_' if FULL else 'tiny_'
MAX_STEPS = None if FULL else 400
EPOCHS = None if FULL else 20
BATCH_SIZE = None if FULL else 16
common = (['--device', 'cuda'] if FULL else ['--tiny', '--device', 'cuda', '--epochs', EPOCHS,
          '--max_steps', MAX_STEPS, '--batch_size', BATCH_SIZE]) + ['--eval_batch_size', 16, '--eval_rotations', 4,
          '--eval_reflections', '--seed', 0]
wanted = [o.strip().upper() for o in OPTIONS_TO_RUN.split(',') if o.strip()]
experiments = [(P+o, 'minilm', o, []) for o in wanted]
if RUN_EXTRAS:
    experiments += [(P+'taskid', 'taskid', 'A', []), (P+'A_legacy', 'minilm', 'A', ['--legacy_odd_scalars'])]
def status():
    for name, *_ in experiments:
        d = SAVE / 'runs' / name
        log = (d / 'log.txt').read_text().splitlines() if (d / 'log.txt').exists() else []
        last = [l for l in log if l.startswith('epoch ')][-1:] or ['(no epochs yet)']
        print(f"{name}: finished={(d / 'results.json').exists()} | last: {last[0]}")
status()
for name, text, option, extra in experiments:
    run_dir = SAVE / 'runs' / name
    if (run_dir / 'results.json').exists() and (run_dir / 'checkpoint.pt').exists():
        print('Completed; skipping', name)
        continue
    run_script('04_train.py', '--resume', '--output_dir', run_dir, '--name', name, '--text', text, '--option', option, *common, *extra)
json.dump(dict(molecules=N_MOLECULES, max_steps=MAX_STEPS, epochs=EPOCHS,
               batch_size=BATCH_SIZE, branch=BRANCH, gpu=torch.cuda.get_device_name()),
          (SAVE / 'config.json').open('w'), indent=2)

status()


In [ ]:
# Re-load each portable checkpoint and independently evaluate rotations + reflections.
done_runs = [e for e in experiments if (SAVE / 'runs' / e[0] / 'results.json').exists()]
for name, _, _, _ in done_runs:
    command = [sys.executable, 'scripts/09_eval_checkpoint.py', '--ckpt',
               str(SAVE / 'runs' / name / 'checkpoint.pt'), '--eval_rotations', '8', '--reflections', '--batch_size', '16']
    with (SAVE / f'{name}_reevaluation.log').open('w') as log:
        subprocess.run(command, check=True, stdout=log, stderr=subprocess.STDOUT)
for name, _, _, _ in done_runs:
    result = json.load(open(SAVE / 'runs' / name / 'results.json'))
    print(name, json.dumps({key: result[key] for key in ['test', 'test_o3', 'test_reflection', 'test_heldout']}))
if RUN_CPU_BENCH:
    run_script('bench_full_model.py', '--threads', 2, '--output', SAVE / 'cpu_full_model_benchmark.json')
shutil.copy2('docs/WORKLOG_oct29b.md', SAVE / 'WORKLOG_oct29b.md')


## TEST results and target checks
The final cell saves the A/B/C comparison beside the cached data and persistent runs.


In [ ]:
# Report TEST force metrics, never validation or held-out wording metrics.
targets = dict(force_mae=0.00727, cosine=0.9446, atom_accuracy=0.754)
summary = dict(split='test', targets=targets, options={})
for option in 'ABC':
    path = SAVE / 'runs' / (P + option) / 'results.json'
    if not path.exists():
        summary['options'][option] = dict(status='not finished')
        continue
    result = json.loads(path.read_text())['test']['force']
    metrics = dict(force_mae=result['mae'], cosine=result['cosine'], atom_accuracy=result['atom_acc'])
    passed = dict(force_mae=metrics['force_mae'] <= targets['force_mae'],
                  cosine=metrics['cosine'] >= targets['cosine'],
                  atom_accuracy=metrics['atom_accuracy'] >= targets['atom_accuracy'])
    summary['options'][option] = dict(metrics=metrics, pass_flags=passed, passed=all(passed.values()))
summary_path = SAVE / ('results_summary.json' if FULL else 'smoke_results_summary.json')
summary_path.write_text(json.dumps(summary, indent=2))
print(summary_path, json.dumps(summary, indent=2))
